<div align="center">
  <img src="assets/Day7.png" alt="Databricks 14 Days AI Challenge - Day 07" width="800"/>
</div>

## DAY 7 (15/01/26) – Workflows & Job Orchestration

### 📚 Learning Objectives
We have built the Bronze, Silver, and Gold layers (Day 6). But running them manually is not scalable. Today, we automate this using **Databricks Workflows**. We will learn:
* **Notebook Widgets:** How to pass dynamic parameters (like "Processing Date" or "Table Name") into a notebook.
* **Multi-Task Orchestration:** Chaining dependencies (Task B starts only if Task A succeeds). 
* **Parameterization:** Using a single codebase to handle different logic branches.

### 🚀 Strategy: The "Controller" Pattern
Instead of maintaining 3 separate notebooks, we will refactor our logic into a single **Parameterized Notebook**.
1.  **Define Widgets:** Create input fields for `pipeline_stage` (Bronze/Silver/Gold).
2.  **Route Logic:** Use Python logic to execute only the code relevant to the selected stage.
3.  **Orchestrate:** In the Databricks UI, we will create a Job with 3 tasks, all pointing to this same notebook but passing different parameters.

###Setup Widgets & Parameters
* **Task**: Add parameter widgets. 
* **Concept**: dbutils.widgets creates UI elements at the top of the notebook. These can be set manually for testing or passed programmatically by the Workflow Job.

In [0]:
# 1. SETUP WIDGETS (UI Inputs)
# "pipeline_stage": Dropdown to select which layer to run
dbutils.widgets.dropdown("pipeline_stage", "bronze", ["bronze", "silver", "gold"])

# "process_date": Text input to simulate processing a specific day's data (Idempotency)
dbutils.widgets.text("process_date", "2019-11-01")

# 2. GET VALUES
current_stage = dbutils.widgets.get("pipeline_stage")
process_date = dbutils.widgets.get("process_date")

print(f"⚙️ JOB CONFIGURATION:")
print(f"   • Current Stage: {current_stage.upper()}")
print(f"   • Processing Date: {process_date}")

# Define global paths based on Volume structure
base_path = "/Volumes/workspace/ecommerce/ecommerce_data"
paths = {
    "raw": f"{base_path}", # CSV source
    "bronze": f"{base_path}/delta/bronze_events",
    "silver": f"{base_path}/delta/silver_events",
    "gold": f"{base_path}/delta/gold_product_perf"
}

###Logic Routing (The Controller)
* **Task**: Define the functions for each layer. 
* **Concept**: We wrap previous days' code into functions. This keeps the notebook clean and modular.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, when, countDistinct, sum, lit

# --- FUNCTION 1: BRONZE LAYER ---
def run_bronze():
    print("🚀 Starting BRONZE Ingestion...")
    # In a real job, we might pick a specific file based on 'process_date'
    df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{paths['raw']}/2019-*.csv")
    
    df_bronze = df_raw.withColumn("ingestion_ts", current_timestamp())
    
    df_bronze.write.format("delta").mode("overwrite").save(paths['bronze'])
    print(f"✅ BRONZE Complete. Written to {paths['bronze']}")

# --- FUNCTION 2: SILVER LAYER ---
def run_silver():
    print("🚀 Starting SILVER Cleaning...")
    df_bronze = spark.read.format("delta").load(paths['bronze'])
    
    df_silver = df_bronze.filter((col("price") > 0) & (col("price") < 50000)) \
                         .dropDuplicates(["user_session", "event_time", "product_id"]) \
                         .withColumn("event_date", to_date("event_time")) \
                         .withColumn("price_tier", when(col("price") < 50, "Budget").otherwise("Premium"))
    
    df_silver.write.format("delta").mode("overwrite").save(paths['silver'])
    print(f"✅ SILVER Complete. Written to {paths['silver']}")

# --- FUNCTION 3: GOLD LAYER (CORRECTED) ---
def run_gold():
    print("🚀 Starting GOLD Aggregation...")
    
    # 1. Read Silver Data
    df_silver = spark.read.format("delta").load(paths['silver'])
    
    # 2. Aggregation Logic (Matching Day 6 Schema)
    df_gold = df_silver.groupBy("product_id", "category_code", "brand") \
        .agg(
            # Count unique users who VIEWED
            countDistinct(when(col("event_type") == "view", col("user_id"))).alias("unique_views"),
            # Count unique users who PURCHASED
            countDistinct(when(col("event_type") == "purchase", col("user_id"))).alias("unique_purchases"),
            # Total Revenue
            sum(when(col("event_type") == "purchase", col("price"))).alias("total_revenue")
        )

    # 3. Add Derived Metric (Conversion Rate)
    df_gold_final = df_gold.withColumn(
        "conversion_rate_pct", 
        (col("unique_purchases") / (col("unique_views") + 1)) * 100
    ).fillna(0)
    
    # 4. Write to Gold (Schema now matches!)
    # We add 'overwriteSchema' just in case there are minor differences, 
    # but the columns above should align perfectly.
    df_gold_final.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(paths['gold'])
        
    print(f"✅ GOLD Complete. Written to {paths['gold']}")

###Execution Block
* **Task**: Execute the logic based on the parameter. 
* **Concept**: This is the "Main" block. It checks the input parameter and runs the corresponding function.

In [0]:
# EXECUTION CONTROLLER
if current_stage == "bronze":
    run_bronze()
elif current_stage == "silver":
    run_silver()
elif current_stage == "gold":
    run_gold()
else:
    raise ValueError(f"❌ Unknown stage: {current_stage}")

### 👷‍♂️ Instructions: Creating the Workflow in Databricks UI
### 🛠️ Architecture
I implemented a **Multi-Task Job** with dependencies:
1.  **Ingest (Bronze):** Ingests raw CSVs.
2.  **Clean (Silver):** Depends on Bronze; validates schema.
3.  **Agg (Gold):** Depends on Silver; calculates business KPIs.

![Job Execution Graph](assets/day_07_success.png)

### 🧠 Key Learnings & Takeaways
* **Modular Design:** By using `dbutils.widgets`, we turned one notebook into a reusable tool for three different pipeline stages.
* **Dependencies:** We learned that Silver must wait for Bronze. In Databricks Jobs, this is handled visually via the **Task Dependency Graph**.
* **Code Reusability:** Instead of maintaining three notebooks (`bronze.ipynb`, `silver.ipynb`, `gold.ipynb`), we maintain one logic hub, reducing technical debt.